# Thermal PV Anomaly Detection — Dataset EDA and Leakage Audit

Dataset: `pkdarabi/solarpanel` (Roboflow export "Anomalies Detection", v14, 8 classes, 640x640 grayscale thermal).

What this notebook produces, in order:
1. Dataset location and inventory of images, labels and annotation format
2. Split inventory (images, source images, augmentation factor)
3. Class distribution, per split, plus the Single / Multi / String taxonomy
4. Objects per image
5. Box geometry (size, elongation, COCO size buckets)
6. Spatial distribution of faults
7. Class co-occurrence
8. Annotated samples per class
9. Pixel statistics and thermal contrast of each fault class
10. Leakage audit (source-level and perceptual-hash)
11. A findings summary written to `/kaggle/working/eda_findings.json`

All figures are saved as 300-dpi PNG and vector PDF in `/kaggle/working/eda_figures`, tables as CSV in `/kaggle/working/eda_tables`.

In [ ]:
import os, re, json, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path(os.environ.get("PV_ROOT", "/kaggle/input/datasets/pkdarabi/solarpanel"))
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./eda_out")
FIG, TAB = OUT / "eda_figures", OUT / "eda_tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

DEFAULT_NAMES = ["MultiByPassed", "MultiDiode", "MultiHotSpot", "SingleByPassed",
                 "SingleDiode", "SingleHotSpot", "StringOpenCircuit", "StringReversedPolarity"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
SPLIT_ALIASES = {"train": ["train"], "valid": ["valid", "val"], "test": ["test"]}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "legend.frameon": False,
})
SPLIT_COLORS = {"train": "#1f77b4", "valid": "#ff7f0e", "test": "#2ca02c"}
FINDINGS = {}


def save(fig, name):
    fig.savefig(FIG / f"{name}.png")
    fig.savefig(FIG / f"{name}.pdf")
    plt.show()

## 1. Locate the dataset and build the image / annotation tables

In [ ]:
if not ROOT.exists():
    search = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
    hits = sorted(search.rglob("data.yaml"))
    assert hits, f"{ROOT} not found and no data.yaml under {search}. Set PV_ROOT or ROOT by hand."
    ROOT = hits[0].parent
print("ROOT:", ROOT)


def find_split_dirs(root):
    found = {}
    for split, names in SPLIT_ALIASES.items():
        cands = [p for n in names for p in root.rglob(n) if p.is_dir() and (p / "images").is_dir()]
        if cands:
            found[split] = min(cands, key=lambda p: len(p.parts))
    return found


SPLIT_DIRS = find_split_dirs(ROOT)
assert SPLIT_DIRS, "No train/valid/test folders with an images/ subfolder were found."
SPLITS = [s for s in ["train", "valid", "test"] if s in SPLIT_DIRS]
for s in SPLITS:
    print(f"{s:>5}: {SPLIT_DIRS[s]}")

NAMES = DEFAULT_NAMES
yamls = sorted(ROOT.rglob("data.yaml"))
if yamls:
    try:
        import yaml
        cfg = yaml.safe_load(open(yamls[0]))
        n = cfg.get("names")
        NAMES = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
        print("data.yaml:", yamls[0], "| license field:", (cfg.get("roboflow") or {}).get("license"))
    except Exception as e:
        print("Could not parse data.yaml, using default names:", e)
NC = len(NAMES)
CLASS_COLORS = {n: plt.cm.tab10(i % 10) for i, n in enumerate(NAMES)}
print("Classes:", NAMES)


def taxonomy(name):
    """MultiHotSpot -> ('Multi', 'HotSpot'). Extent of the fault x physical fault type."""
    m = re.match(r"^(Single|Multi|String)(.+)$", name)
    return (m.group(1), m.group(2)) if m else ("Other", name)


def source_id(stem):
    """Roboflow names files '<original>_jpg.rf.<hash>'. Augmented copies share the part before '.rf.'."""
    return re.split(r"\.rf\.", stem)[0]


def parse_label(path):
    """Returns (rows, n_bad). Each row: cls, xc, yc, w, h, n_pts (0 for a plain box, k for a k-point polygon)."""
    rows, bad = [], 0
    if not path.exists():
        return rows, bad
    for line in path.read_text().strip().splitlines():
        p = line.split()
        if not p:
            continue
        try:
            c, v = int(float(p[0])), np.array(p[1:], dtype=float)
        except ValueError:
            bad += 1
            continue
        if len(v) == 4:
            xc, yc, w, h, k = *v, 0
        elif len(v) >= 6 and len(v) % 2 == 0:
            xs, ys = v[0::2], v[1::2]
            xc, yc = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
            w, h, k = xs.max() - xs.min(), ys.max() - ys.min(), len(xs)
        else:
            bad += 1
            continue
        if w <= 0 or h <= 0 or not (0 <= c < NC):
            bad += 1
            continue
        rows.append((c, xc, yc, w, h, k))
    return rows, bad


img_rows, ann_rows, n_bad_total = [], [], 0
for split in SPLITS:
    idir, ldir = SPLIT_DIRS[split] / "images", SPLIT_DIRS[split] / "labels"
    files = sorted(p for p in idir.iterdir() if p.suffix.lower() in IMG_EXT)
    for p in tqdm(files, desc=f"indexing {split}"):
        lp = ldir / f"{p.stem}.txt"
        rows, bad = parse_label(lp)
        n_bad_total += bad
        try:
            with Image.open(p) as im:
                W, H, mode = im.size[0], im.size[1], im.mode
        except Exception:
            W = H = 0
            mode = "corrupt"
        sid = source_id(p.stem)
        img_rows.append(dict(split=split, path=str(p), file=p.name, source_id=sid, has_rf=".rf." in p.stem,
                             W=W, H=H, mode=mode, has_label=lp.exists(), n_obj=len(rows),
                             n_cls=len({r[0] for r in rows})))
        for c, xc, yc, w, h, k in rows:
            ann_rows.append(dict(split=split, path=str(p), source_id=sid, cls=c, name=NAMES[c],
                                 xc=xc, yc=yc, w=w, h=h, n_pts=k, W=W, H=H))

img_df = pd.DataFrame(img_rows)
ann = pd.DataFrame(ann_rows)
ann["extent"], ann["fault"] = zip(*ann["name"].map(taxonomy))
ann["area"] = ann.w * ann.h                                   # fraction of image area
ann["area_px"] = ann.area * ann.W * ann.H
ann["elong"] = np.maximum(ann.w, ann.h) / np.minimum(ann.w, ann.h)   # orientation-free, since rot90 is baked in
ann["size_bucket"] = pd.cut(ann.area_px, [0, 32 ** 2, 96 ** 2, np.inf], labels=["small", "medium", "large"])

fmt = "polygon (segmentation)" if (ann.n_pts > 0).mean() > 0.5 else "bounding box"
print(f"\n{len(img_df):,} images | {len(ann):,} instances | label format: {fmt} | malformed label lines: {n_bad_total}")
print("Image sizes:", dict(Counter(zip(img_df.W, img_df.H)).most_common(3)), "| modes:", dict(img_df["mode"].value_counts()))
FINDINGS["inventory"] = dict(images=int(len(img_df)), instances=int(len(ann)), label_format=fmt,
                             polygon_share=float((ann.n_pts > 0).mean()), malformed_lines=int(n_bad_total),
                             corrupt_images=int((img_df["mode"] == "corrupt").sum()))

## 2. Split inventory
`sources` counts unique pre-augmentation images. `aug_factor` should be about 3 for train and 1 for valid/test if the README is accurate.

In [ ]:
split_tbl = img_df.groupby("split").agg(
    images=("file", "size"), sources=("source_id", "nunique"),
    background=("n_obj", lambda s: int((s == 0).sum())),
    missing_label=("has_label", lambda s: int((~s).sum())),
    instances=("n_obj", "sum")).reindex(SPLITS)
split_tbl["share_%"] = (100 * split_tbl.images / split_tbl.images.sum()).round(1)
split_tbl["aug_factor"] = (split_tbl.images / split_tbl.sources).round(2)
split_tbl["inst_per_img"] = (split_tbl.instances / split_tbl.images).round(2)
split_tbl.to_csv(TAB / "split_summary.csv")
display(split_tbl)
FINDINGS["splits"] = json.loads(split_tbl.to_json(orient="index"))
FINDINGS["unique_sources_total"] = int(img_df.source_id.nunique())

## 3. Class distribution

In [ ]:
cls_tbl = ann.pivot_table(index="name", columns="split", values="cls", aggfunc="size", fill_value=0)
cls_tbl = cls_tbl.reindex(NAMES).fillna(0).astype(int)[SPLITS]
cls_tbl["total"] = cls_tbl.sum(axis=1)
cls_tbl["share_%"] = (100 * cls_tbl.total / cls_tbl.total.sum()).round(2)
cls_tbl["images"] = ann.groupby("name").path.nunique().reindex(NAMES).fillna(0).astype(int)
# Class-balanced weights from the effective number of samples (Cui et al., CVPR 2019), beta = 0.999
beta, n_tr = 0.999, cls_tbl["train" if "train" in cls_tbl else "total"].clip(lower=1)
w_eff = (1 - beta) / (1 - beta ** n_tr)
cls_tbl["cb_weight"] = (w_eff / w_eff.sum() * NC).round(3)
cls_tbl.to_csv(TAB / "class_distribution.csv")
display(cls_tbl)

share = cls_tbl[SPLITS] / cls_tbl[SPLITS].sum(axis=0)
nz = cls_tbl.total[cls_tbl.total > 0]
imb = float(nz.max() / nz.min())
drift = {s: float((share[s] - share["train"]).abs().max() * 100) for s in SPLITS if s != "train"} if "train" in SPLITS else {}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"width_ratios": [1.25, 1.25, 1]})
y, bh = np.arange(NC), 0.8 / len(SPLITS)
for i, s in enumerate(SPLITS):
    axes[0].barh(y + i * bh, cls_tbl[s], bh, label=s, color=SPLIT_COLORS[s])
    axes[1].barh(y + i * bh, 100 * share[s], bh, label=s, color=SPLIT_COLORS[s])
for a, t, xl in [(axes[0], "Instances per class", "instances (log scale)"),
                 (axes[1], "Class share within each split", "% of the split's instances")]:
    a.set_yticks(y + bh * (len(SPLITS) - 1) / 2)
    a.set_yticklabels(NAMES)
    a.invert_yaxis()
    a.set_title(t)
    a.set_xlabel(xl)
axes[0].set_xscale("log")
axes[1].set_yticklabels([])
axes[1].legend(loc="lower right")

tax = ann.pivot_table(index="extent", columns="fault", values="cls", aggfunc="size", fill_value=0)
im = axes[2].imshow(np.log10(tax.values + 1), cmap="YlOrRd", aspect="auto")
axes[2].set_xticks(range(tax.shape[1]))
axes[2].set_xticklabels(tax.columns, rotation=35, ha="right")
axes[2].set_yticks(range(tax.shape[0]))
axes[2].set_yticklabels(tax.index)
for i in range(tax.shape[0]):
    for j in range(tax.shape[1]):
        v = tax.values[i, j]
        axes[2].text(j, i, f"{v:,}" if v else "n/a", ha="center", va="center", fontsize=9,
                     color="black" if v else "grey")
axes[2].set_title("Taxonomy: extent x fault type")
axes[2].grid(False)
fig.suptitle(f"Class distribution (imbalance ratio {imb:.1f}:1)", y=1.02, fontsize=12)
save(fig, "fig01_class_distribution")

tax.to_csv(TAB / "taxonomy_counts.csv")
FINDINGS["classes"] = dict(imbalance_ratio=imb, largest=str(nz.idxmax()), smallest=str(nz.idxmin()),
                           counts=cls_tbl.total.to_dict(), share_pct=cls_tbl["share_%"].to_dict(),
                           max_share_drift_vs_train_pct=drift,
                           valid_taxonomy_cells=int((tax.values > 0).sum()), taxonomy_cells=int(tax.size))

## 4. Objects per image

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cap = int(max(5, img_df.n_obj.quantile(0.99)))
bins = np.arange(-0.5, cap + 1.5, 1)
for s in SPLITS:
    axes[0].hist(img_df[img_df.split == s].n_obj.clip(upper=cap), bins=bins, density=True, histtype="step",
                 lw=2, label=s, color=SPLIT_COLORS[s])
axes[0].set(title="Instances per image", xlabel=f"instances (clipped at {cap}, the 99th percentile)", ylabel="density")
axes[0].legend()
vc = img_df.n_cls.value_counts(normalize=True).sort_index() * 100
axes[1].bar(vc.index.astype(str), vc.values, color="#555")
for i, v in enumerate(vc.values):
    axes[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
axes[1].set(title="Distinct classes per image", xlabel="number of distinct classes", ylabel="% of images")
save(fig, "fig02_objects_per_image")

q = img_df.n_obj.describe(percentiles=[.5, .9, .99])
FINDINGS["density"] = dict(mean=float(q["mean"]), median=float(q["50%"]), p90=float(q["90%"]), p99=float(q["99%"]),
                           max=int(q["max"]), background_pct=float((img_df.n_obj == 0).mean() * 100),
                           multi_class_image_pct=float((img_df.n_cls > 1).mean() * 100))
print(q.round(2).to_string())

## 5. Box geometry
Elongation is `max(w, h) / min(w, h)`, so it is unaffected by the 90-degree rotations already baked into the training split.
Size buckets follow COCO: small < 32^2 px, medium < 96^2 px, large otherwise, at the stored 640x640 resolution.

In [ ]:
geo = ann.groupby("name").agg(n=("cls", "size"), w_med=("w", "median"), h_med=("h", "median"),
                              area_pct_med=("area", lambda s: 100 * s.median()),
                              elong_med=("elong", "median")).reindex(NAMES)
bucket = (ann.pivot_table(index="name", columns="size_bucket", values="cls", aggfunc="size", fill_value=0)
          .reindex(NAMES).fillna(0))
bucket_pct = 100 * bucket.div(bucket.sum(axis=1).replace(0, 1), axis=0)
geo = geo.join(bucket_pct.round(1).add_suffix("_%"))
geo.round(3).to_csv(TAB / "box_geometry.csv")
display(geo.round(3))

present = [n for n in NAMES if (ann.name == n).any()]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
hb = axes[0, 0].hexbin(ann.w, ann.h, gridsize=45, bins="log", cmap="viridis", extent=(0, 1, 0, 1))
axes[0, 0].set(title="Box width vs height (normalised)", xlabel="width", ylabel="height")
fig.colorbar(hb, ax=axes[0, 0], label="count (log)")
for a, col, ttl, yl in [(axes[0, 1], "area", "Box area per class", "% of image area (log)"),
                        (axes[1, 0], "elong", "Box elongation per class", "max side / min side (log)")]:
    data = [(ann[ann.name == n][col] * (100 if col == "area" else 1)).values for n in present]
    bp = a.boxplot(data, vert=True, patch_artist=True, showfliers=False)
    for patch, n in zip(bp["boxes"], present):
        patch.set_facecolor(CLASS_COLORS[n])
        patch.set_alpha(0.75)
    a.set_xticklabels(present, rotation=35, ha="right")
    a.set_yscale("log")
    a.set(title=ttl, ylabel=yl)
left = np.zeros(NC)
for b, c in zip(["small", "medium", "large"], ["#d62728", "#ff7f0e", "#2ca02c"]):
    vals = bucket_pct[b].values if b in bucket_pct else np.zeros(NC)
    axes[1, 1].barh(NAMES, vals, left=left, label=b, color=c)
    left += vals
axes[1, 1].invert_yaxis()
axes[1, 1].set(title="COCO size buckets per class", xlabel="% of the class's instances")
axes[1, 1].legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.28))
fig.tight_layout()
save(fig, "fig03_box_geometry")

FINDINGS["geometry"] = dict(
    small_pct=float((ann.size_bucket == "small").mean() * 100),
    medium_pct=float((ann.size_bucket == "medium").mean() * 100),
    large_pct=float((ann.size_bucket == "large").mean() * 100),
    median_area_pct_by_class=geo.area_pct_med.round(3).to_dict(),
    median_elongation_by_class=geo.elong_med.round(2).to_dict(),
    area_ratio_largest_to_smallest_class=float(geo.area_pct_med.max() / max(geo.area_pct_med.min(), 1e-9)))

## 6. Where faults sit in the frame

In [ ]:
ncol = 4
nrow = int(np.ceil(NC / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.5 * nrow))
for a, n in zip(np.ravel(axes), NAMES):
    d = ann[ann.name == n]
    a.hist2d(d.xc, d.yc, bins=32, range=[[0, 1], [0, 1]], cmap="magma")
    a.invert_yaxis()
    a.set_aspect("equal")
    a.set_title(f"{n} (n={len(d):,})", fontsize=10)
    a.set_xticks([])
    a.set_yticks([])
    a.grid(False)
for a in np.ravel(axes)[NC:]:
    a.axis("off")
fig.suptitle("Density of box centres per class", y=1.0)
fig.tight_layout()
save(fig, "fig04_spatial_heatmaps")

## 7. Class co-occurrence
Cell (row i, column j) is P(class j appears in an image | class i appears in it).

In [ ]:
M = (ann.assign(v=1).pivot_table(index="path", columns="name", values="v", aggfunc="max", fill_value=0)
     .reindex(columns=NAMES, fill_value=0).values.astype(float))
co = M.T @ M
diag = np.diag(co).copy()
cond = co / np.where(diag == 0, 1, diag)[:, None]
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cond, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NC))
ax.set_xticklabels(NAMES, rotation=40, ha="right")
ax.set_yticks(range(NC))
ax.set_yticklabels(NAMES)
for i in range(NC):
    for j in range(NC):
        ax.text(j, i, f"{cond[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cond[i, j] > 0.55 else "black")
ax.grid(False)
ax.set_title("Image-level co-occurrence, P(column | row)")
fig.colorbar(im, ax=ax, fraction=0.046)
save(fig, "fig05_cooccurrence")
pd.DataFrame(cond, index=NAMES, columns=NAMES).round(3).to_csv(TAB / "cooccurrence.csv")
off = cond - np.eye(NC)
i, j = np.unravel_index(np.argmax(off), off.shape)
FINDINGS["cooccurrence"] = dict(strongest_pair=[NAMES[i], NAMES[j]], p_col_given_row=float(off[i, j]))

## 8. Annotated samples
Drawn from valid/test when possible, because those images carry no augmentation.

In [ ]:
def draw(ax, path, focus=None):
    im = Image.open(path).convert("RGB")
    W, H = im.size
    ax.imshow(im)
    for r in ann[ann.path == path].itertuples():
        x0, y0 = (r.xc - r.w / 2) * W, (r.yc - r.h / 2) * H
        ax.add_patch(mpatches.Rectangle((x0, y0), r.w * W, r.h * H, fill=False, lw=1.8 if r.name == focus else 1.0,
                                        ec=CLASS_COLORS[r.name]))
    ax.axis("off")


K = 4
fig, axes = plt.subplots(NC, K, figsize=(3.1 * K, 3.1 * NC))
rng = random.Random(SEED)
for row, n in enumerate(NAMES):
    pool = ann[(ann.name == n) & (ann.split != "train")].path.unique().tolist()
    if len(pool) < K:
        pool += ann[(ann.name == n) & (ann.split == "train")].path.unique().tolist()
    picks = rng.sample(pool, min(K, len(pool)))
    for col in range(K):
        a = axes[row, col]
        if col < len(picks):
            draw(a, picks[col], focus=n)
        else:
            a.axis("off")
        if col == 0:
            a.set_title(n, loc="left", fontsize=11, color=CLASS_COLORS[n], fontweight="bold")
fig.legend(handles=[mpatches.Patch(fc="none", ec=CLASS_COLORS[n], label=n, lw=2) for n in NAMES],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.03, 1, 1))
save(fig, "fig06_samples_per_class")

## 9. Pixel statistics and thermal contrast
For every box in a stratified sample of images, contrast is the mean intensity inside the box minus the mean of a surrounding ring
(the box enlarged by 50% on each side, minus the box), on a 0-1 scale. Positive means the fault is hotter than its neighbourhood.
A class whose contrast sits near zero is one the detector has to recognise from shape and context, not brightness.

In [ ]:
N_PER_SPLIT = 500
sample = pd.concat([g.sample(min(N_PER_SPLIT, len(g)), random_state=SEED) for _, g in img_df.groupby("split")])
ann_by_path = {p: g for p, g in ann.groupby("path")}
hist = {s: np.zeros(256) for s in SPLITS}
px_rows, ct_rows, chan_diff = [], [], []
for r in tqdm(sample.itertuples(), total=len(sample), desc="pixel stats"):
    try:
        arr = np.asarray(Image.open(r.path))
    except Exception:
        continue
    if arr.ndim == 3:
        a3 = arr[..., :3].astype(np.int16)
        chan_diff.append(int(np.abs(a3 - a3.mean(axis=2, keepdims=True)).max()))
        g = a3.mean(axis=2)
    else:
        chan_diff.append(0)
        g = arr.astype(np.float32)
    H, W = g.shape
    hist[r.split] += np.bincount(g.astype(np.uint8).ravel(), minlength=256)
    px_rows.append(dict(split=r.split, mean=g.mean() / 255, std=g.std() / 255))
    for b in ann_by_path.get(r.path, pd.DataFrame()).itertuples():
        x0, x1 = int(max(0, (b.xc - b.w / 2) * W)), int(min(W, (b.xc + b.w / 2) * W))
        y0, y1 = int(max(0, (b.yc - b.h / 2) * H)), int(min(H, (b.yc + b.h / 2) * H))
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        mx, my = int(0.5 * (x1 - x0)), int(0.5 * (y1 - y0))
        X0, X1, Y0, Y1 = max(0, x0 - mx), min(W, x1 + mx), max(0, y0 - my), min(H, y1 + my)
        inner, outer = g[y0:y1, x0:x1], g[Y0:Y1, X0:X1]
        ring_n = outer.size - inner.size
        if ring_n <= 0:
            continue
        ring_mean = (outer.sum() - inner.sum()) / ring_n
        ct_rows.append(dict(name=b.name, split=r.split, contrast=(inner.mean() - ring_mean) / 255,
                            inside_mean=inner.mean() / 255, inside_std=inner.std() / 255))
px, ct = pd.DataFrame(px_rows), pd.DataFrame(ct_rows)
is_gray = bool(np.percentile(chan_diff, 99) <= 3) if chan_diff else None
print("Effectively single-channel (R=G=B within JPEG tolerance):", is_gray)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4), gridspec_kw={"width_ratios": [1, 0.8, 1.3]})
for s in SPLITS:
    if hist[s].sum():
        axes[0].plot(hist[s] / hist[s].sum(), label=s, color=SPLIT_COLORS[s], lw=1.6)
axes[0].set(title="Intensity histogram by split", xlabel="grey level", ylabel="share of pixels")
axes[0].legend()
bp = axes[1].boxplot([px[px.split == s]["mean"].values for s in SPLITS], patch_artist=True, showfliers=False)
for patch, s in zip(bp["boxes"], SPLITS):
    patch.set_facecolor(SPLIT_COLORS[s])
    patch.set_alpha(0.7)
axes[1].set_xticklabels(SPLITS)
axes[1].set(title="Mean image brightness by split", ylabel="mean intensity (0-1)")
present_ct = [n for n in NAMES if len(ct) and (ct.name == n).any()]
bp = axes[2].boxplot([ct[ct.name == n].contrast.values for n in present_ct], patch_artist=True, showfliers=False)
for patch, n in zip(bp["boxes"], present_ct):
    patch.set_facecolor(CLASS_COLORS[n])
    patch.set_alpha(0.75)
axes[2].axhline(0, color="k", lw=0.8, ls="--")
axes[2].set_xticklabels(present_ct, rotation=35, ha="right")
axes[2].set(title="Thermal contrast: box vs surrounding ring", ylabel="mean(inside) - mean(ring), 0-1 scale")
save(fig, "fig07_pixel_stats_contrast")

ct_tbl = ct.groupby("name").agg(n=("contrast", "size"), contrast_med=("contrast", "median"),
                                contrast_iqr=("contrast", lambda s: s.quantile(.75) - s.quantile(.25)),
                                negative_share=("contrast", lambda s: (s < 0).mean()),
                                inside_mean=("inside_mean", "median"), inside_std=("inside_std", "median")
                                ).reindex(NAMES) if len(ct) else pd.DataFrame()
ct_tbl.round(4).to_csv(TAB / "thermal_contrast.csv")
display(ct_tbl.round(4))
px_split = px.groupby("split")[["mean", "std"]].mean().reindex(SPLITS)
FINDINGS["pixels"] = dict(
    single_channel=is_gray, dataset_mean=float(px["mean"].mean()), dataset_std=float(px["std"].mean()),
    mean_brightness_by_split=px_split["mean"].round(4).to_dict(),
    contrast_median_by_class=ct_tbl.contrast_med.round(4).to_dict() if len(ct_tbl) else {},
    lowest_contrast_class=str(ct_tbl.contrast_med.abs().idxmin()) if len(ct_tbl) and ct_tbl.contrast_med.notna().any() else None,
    highest_contrast_class=str(ct_tbl.contrast_med.idxmax()) if len(ct_tbl) and ct_tbl.contrast_med.notna().any() else None)

## 10. Leakage audit

**A. Source level.** Roboflow augments after splitting, so no source image should appear in two splits. This checks it from the file names.

**B. Scene level.** Drone thermography is shot as overlapping frames, so two *different* source images can show the same panels.
A random split then puts near-identical scenes in train and test. Each image is reduced to a 16x16 median-thresholded hash (256 bits,
insensitive to global brightness), compared under all 8 flips and 90-degree rotations.
Augmented copies of the same training source give a reference for what "the same scene" looks like under this hash, and random pairs of different
sources give the "unrelated" reference. Cross-split nearest neighbours that fall in the same-scene range are candidates for manual review. This is a screen and
does not prove leakage on its own, and shear makes it conservative.

In [ ]:
rf_cov = float(img_df.has_rf.mean())
src_splits = img_df.groupby("source_id").split.nunique()
leaky = src_splits[src_splits > 1]
print(f"Roboflow naming pattern found in {rf_cov:.1%} of files.")
print(f"Source images present in more than one split: {len(leaky)} of {len(src_splits):,}")
if len(leaky):
    display(img_df[img_df.source_id.isin(leaky.index)].sort_values("source_id")[["source_id", "split", "file"]].head(20))
ver = img_df.groupby(["split", "source_id"]).size().groupby("split").agg(["mean", "min", "max"]).reindex(SPLITS)
print("\nAugmented versions per source image:")
print(ver.round(2).to_string())
prefix = img_df.source_id.str.replace(r"[\d_\-]+(?:jpg|jpeg|png)?$", "", regex=True, flags=re.I)
print("\nMost common file-name prefixes (possible flights or sites, useful for a group-wise split):")
print(prefix.value_counts().head(8).to_string())
print("\nExample file names:", img_df.file.sample(min(4, len(img_df)), random_state=SEED).tolist())

In [ ]:
HS = 16
RES = getattr(Image, "Resampling", Image).BILINEAR


def dihedral(a):
    out = []
    for k in range(4):
        r = np.rot90(a, k)
        out += [r, np.fliplr(r)]
    return out


def hash8(path):
    try:
        with Image.open(path) as im:
            im.draft("L", (HS * 4, HS * 4))
            a = np.asarray(im.convert("L").resize((HS, HS), RES), dtype=np.float32)
    except Exception:
        return None
    return np.stack([(t > np.median(t)).ravel() for t in dihedral(a)])


_h = [hash8(p) for p in tqdm(img_df.path, desc="hashing")]
VALID = np.array([h is not None for h in _h])                               # unreadable images are left out of every comparison
Hh = np.stack([h if h is not None else np.zeros((8, HS * HS), dtype=bool) for h in _h])      # [N, 8, 256]
idx = {s: np.where((img_df.split.values == s) & VALID)[0] for s in SPLITS}
print(f"Hashed {int(VALID.sum()):,} of {len(VALID):,} images; {int((~VALID).sum())} unreadable images excluded.")


def nn_dist(a_idx, b_idx):
    """For every image in A: Hamming distance to, and index of, its nearest image in B over the 8 transforms."""
    A = Hh[a_idx, 0].astype(np.float32)
    best = np.full(len(a_idx), np.inf, dtype=np.float32)
    arg = np.zeros(len(a_idx), dtype=int)
    for t in range(8):
        B = Hh[b_idx, t].astype(np.float32)
        d = A @ (1 - B).T + (1 - A) @ B.T
        m, am = d.min(axis=1), d.argmin(axis=1)
        upd = m < best
        best[upd], arg[upd] = m[upd], am[upd]
    return best, b_idx[arg]


def pair_dist(i, j):
    return (Hh[i, 0][:, None, :] != Hh[j]).sum(axis=2).min(axis=1)


# Reference 1: augmented copies of the same source inside train (known same scene)
tr = img_df[(img_df.split == "train") & VALID] if "train" in SPLITS else img_df.iloc[:0]
same_i, same_j = [], []
for _, g in tr.groupby("source_id"):
    ii = g.index.values
    if len(ii) > 1:
        same_i.append(ii[0])
        same_j.append(ii[1])
    if len(same_i) >= 2000:
        break
d_same = pair_dist(np.array(same_i), np.array(same_j)) if same_i else np.array([])
# Reference 2: random pairs from different sources
rs = np.random.RandomState(SEED)
N_RAND = 60000
ri, rj = rs.randint(0, len(img_df), N_RAND), rs.randint(0, len(img_df), N_RAND)
keep = (img_df.source_id.values[ri] != img_df.source_id.values[rj]) & VALID[ri] & VALID[rj]
ri, rj = ri[keep], rj[keep]
d_rand = np.concatenate([pair_dist(ri[k:k + 4000], rj[k:k + 4000]) for k in range(0, len(ri), 4000)])

THR = float(np.ceil(np.percentile(d_same, 90))) if len(d_same) else 0.05 * HS * HS
# How often does an unrelated pair fall under the threshold by chance? A query is compared with every image of the
# other split, so the expected number of chance hits per query is that rate times the size of the other split.
fpr = float((d_rand <= THR).mean())
chance_hits = fpr * max(len(v) for v in idx.values())
informative = chance_hits < 0.2
pairs, cross = [("valid", "train"), ("test", "train"), ("test", "valid")], {}
for a, b in pairs:
    if a in idx and b in idx and len(idx[a]) and len(idx[b]):
        d, j = nn_dist(idx[a], idx[b])
        cross[(a, b)] = (d, j)

fig, ax = plt.subplots(figsize=(10, 4.4))
bins = np.arange(0, HS * HS // 2 + 4, 4)
if len(d_same):
    ax.hist(d_same, bins=bins, density=True, alpha=0.35, color="crimson", label="same source, augmented copies (train)")
ax.hist(d_rand, bins=bins, density=True, alpha=0.35, color="grey", label="random pairs, different sources")
for (a, b), (d, _) in cross.items():
    ax.hist(d, bins=bins, density=True, histtype="step", lw=2, label=f"{a} image -> nearest {b} image")
ax.axvline(THR, color="k", ls="--", lw=1)
ax.annotate("review threshold", (THR, 1.0), xycoords=("data", "axes fraction"), fontsize=8, ha="center", va="bottom")
ax.set(title="Perceptual-hash distance (256-bit, flip/rotation invariant)", xlabel="Hamming distance", ylabel="density")
ax.legend(fontsize=8)
save(fig, "fig08_leakage_hash_distances")

leak = {}
for (a, b), (d, _) in cross.items():
    leak[f"{a}_vs_{b}"] = dict(n=int(len(d)), exact=int((d == 0).sum()), below_threshold=int((d <= THR).sum()),
                               below_threshold_pct=float((d <= THR).mean() * 100), median_nn=float(np.median(d)))
print(f"Review threshold = {THR:.0f} bits (90th percentile of the distance between augmented copies of the same source)")
print(f"Unrelated pairs under the threshold: {fpr:.4%} -> about {chance_hits:.2f} chance hits per query image.")
print("Screen is", "INFORMATIVE: counts below are meaningful." if informative else
      "NOT INFORMATIVE on its own: scenes look too alike for this hash, judge from the image pairs in the next cell.")
print(pd.DataFrame(leak).T.to_string())
FINDINGS["leakage"] = dict(rf_pattern_coverage=rf_cov, sources_in_multiple_splits=int(len(leaky)),
                           versions_per_source=json.loads(ver.round(2).to_json(orient="index")),
                           hash_threshold_bits=THR, chance_hits_per_query=chance_hits, hash_screen_informative=bool(informative),
                           same_source_median=float(np.median(d_same)) if len(d_same) else None,
                           same_source_p90=float(np.percentile(d_same, 90)) if len(d_same) else None,
                           random_pair_median=float(np.median(d_rand)), cross_split=leak)

### Closest cross-split pairs
Look at these. If the left and right images show the same panels, the official split leaks scenes and the paper needs a group-wise re-split.

In [ ]:
cands = []
for (a, b), (d, j) in cross.items():
    for k in np.argsort(d)[:6]:
        cands.append((float(d[k]), idx[a][k], j[k], a, b))
cands = sorted(cands)[:8]
if cands:
    fig, axes = plt.subplots(2, len(cands), figsize=(2.6 * len(cands), 5.6))
    axes = np.atleast_2d(axes).reshape(2, -1)
    for c, (d, ia, ib, a, b) in enumerate(cands):
        for rrow, (ii, s) in enumerate([(ia, a), (ib, b)]):
            axes[rrow, c].imshow(Image.open(img_df.path[ii]).convert("RGB"))
            axes[rrow, c].axis("off")
            axes[rrow, c].set_title(f"{s}" + (f" | d={d:.0f}" if rrow == 0 else ""), fontsize=9)
    fig.suptitle("Nearest cross-split pairs (top row vs bottom row)", y=1.0)
    fig.tight_layout()
    save(fig, "fig09_leakage_closest_pairs")
pd.DataFrame([dict(distance=d, split_a=a, file_a=img_df.file[ia], split_b=b, file_b=img_df.file[ib])
              for d, ia, ib, a, b in cands]).to_csv(TAB / "closest_cross_split_pairs.csv", index=False)

## 11. Findings summary
Copy the printed block (or attach `eda_findings.json`) back into the chat. The model design is decided from these numbers.

In [ ]:
F = FINDINGS
L = []
sp = F["splits"]
L.append(f"Inventory: {F['inventory']['images']:,} images, {F['inventory']['instances']:,} instances, "
         f"{F['unique_sources_total']:,} unique source images; labels are {F['inventory']['label_format']} "
         f"({F['inventory']['polygon_share']:.0%} polygons); {F['inventory']['malformed_lines']} malformed lines, "
         f"{F['inventory']['corrupt_images']} unreadable images.")
L.append("Splits: " + "; ".join(f"{s} {int(v['images']):,} img ({v['share_%']}%, x{v['aug_factor']} per source, "
                                 f"{int(v['background'])} background)" for s, v in sp.items()) + ".")
c = F["classes"]
L.append(f"Imbalance: {c['imbalance_ratio']:.1f}:1 between {c['largest']} ({c['counts'][c['largest']]:,}) and "
         f"{c['smallest']} ({c['counts'][c['smallest']]:,}). Largest class-share gap vs train: "
         + ", ".join(f"{s} {v:.1f} pts" for s, v in c["max_share_drift_vs_train_pct"].items()) + ".")
L.append(f"Taxonomy: {c['valid_taxonomy_cells']} of {c['taxonomy_cells']} extent x fault combinations occur.")
d = F["density"]
L.append(f"Density: mean {d['mean']:.2f}, median {d['median']:.0f}, p99 {d['p99']:.0f}, max {d['max']} instances per image; "
         f"{d['background_pct']:.1f}% background images; {d['multi_class_image_pct']:.1f}% of images hold more than one class.")
g = F["geometry"]
L.append(f"Object size: {g['small_pct']:.1f}% small, {g['medium_pct']:.1f}% medium, {g['large_pct']:.1f}% large; "
         f"median box area differs {g['area_ratio_largest_to_smallest_class']:.0f}x between the largest and smallest class.")
co_ = F["cooccurrence"]
L.append(f"Co-occurrence: strongest is P({co_['strongest_pair'][1]} | {co_['strongest_pair'][0]}) = {co_['p_col_given_row']:.2f}.")
p = F["pixels"]
L.append(f"Pixels: single-channel = {p['single_channel']}; mean {p['dataset_mean']:.3f}, std {p['dataset_std']:.3f}; "
         f"brightness by split {p['mean_brightness_by_split']}. Highest thermal contrast: {p['highest_contrast_class']}; "
         f"closest to zero: {p['lowest_contrast_class']}.")
lk = F["leakage"]
L.append(f"Leakage A: {lk['sources_in_multiple_splits']} source images appear in more than one split "
         f"(naming pattern covers {lk['rf_pattern_coverage']:.0%} of files).")
L.append("Leakage B: " + "; ".join(f"{k}: {v['below_threshold']}/{v['n']} ({v['below_threshold_pct']:.1f}%) within "
                                    f"{lk['hash_threshold_bits']:.0f} bits, {v['exact']} exact"
                                    for k, v in lk["cross_split"].items())
         + f". Reference distances: same source median {lk['same_source_median']} / p90 {lk['same_source_p90']}, random pairs median {lk['random_pair_median']:.0f}. "
         + ("Screen informative." if lk["hash_screen_informative"] else
            f"Screen NOT informative ({lk['chance_hits_per_query']:.1f} chance hits per query); use the image pairs."))

print("=" * 100)
print("EDA FINDINGS")
print("=" * 100)
for k, s in enumerate(L, 1):
    print(f"{k:>2}. {s}")
json.dump(F, open(OUT / "eda_findings.json", "w"), indent=2, default=str)
print("\nSaved:", OUT / "eda_findings.json", "|", len(list(FIG.glob('*.png'))), "figures in", FIG, "|",
      len(list(TAB.glob('*.csv'))), "tables in", TAB)